# 당뇨 예측 - CatBoost Feature Engineering 실험

- 타겟: `당뇨유병` (0: 없음 / 1: 있음)
- 모델: CatBoost (Optuna 최적 파라미터 고정)
- Threshold: 0.50 고정
- 목적: **피처 추가·제거 실험으로 성능 변화 확인**
- 검증: Stratified 5-Fold CV

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, f1_score, recall_score, precision_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
)
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

# ── 경로 설정 ──────────────────────────────────
CURRENT_DIR = os.path.dirname(os.path.abspath(''))
BASE_DIR = os.path.join(CURRENT_DIR, '..')
DATA_PATH    = os.path.join(BASE_DIR, 'data', 'v0.1_x1_preprocessed.csv')
FEATURES_DIR = os.path.join(BASE_DIR, 'features')
NPY_DIR      = os.path.join(BASE_DIR, 'outputs', 'oof')
sys.path.insert(0, FEATURES_DIR)

RANDOM_STATE = 42
THRESHOLD    = 0.50

## 1. 데이터 로드

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'로드 완료 | shape: {df.shape}')

## 2. ✏️ Feature Engineering 설정


In [ ]:
# ── 피처 엔지니어링 함수 import ──────────────────────────────
from fe_age_bin       import add_age_bin           # 나이_구간

# ── 적용할 피처 선택  ──────────────────────
df = add_age_bin(df)                    # ✅ 나이_구간

print(f'\n피처 엔지니어링 완료 | shape: {df.shape}')

## 3. 피처 / 타겟 분리

In [ ]:
TARGET    = '당뇨유병'
DROP_COLS = ['당뇨유병', '당뇨유병', '이상지질혈증유병', '비만단계']

data  = df.dropna(subset=[TARGET]).copy()
X     = data.drop(columns=DROP_COLS)
y     = data[TARGET].astype(int)
neg, pos = (y == 0).sum(), (y == 1).sum()
ratio = neg / pos

print(f'샘플 수: {len(y)}  |  정상: {neg}  |  고혈압: {pos}')
print(f'사용 피처 수: {X.shape[1]}')
print(f'피처 목록: {list(X.columns)}')

## 4. Optuna 최적 파라미터 설정

In [ ]:
best_params = dict(
    iterations          = 526,
    learning_rate       = 0.02730273608044661,
    depth               = 5,
    l2_leaf_reg         = 7.860835985184405,
    bagging_temperature = 0.5392727789146318,
    random_strength     = 0.5400950201135939,
    border_count        = 214,
    loss_function         = 'Logloss',
    eval_metric           = 'AUC',
    class_weights         = {0: 1.0, 1: ratio},
    early_stopping_rounds = 50,
    random_seed           = RANDOM_STATE,
    verbose               = False,
    allow_writing_files   = False,
)
print('파라미터 설정 완료')

## 5. Stratified 5-Fold CV

In [ ]:
skf         = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
oof_proba   = np.zeros(len(y))
fold_scores = []

print('=' * 65)
for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
    model = CatBoostClassifier(**best_params)
    model.fit(Pool(X_tr, y_tr), eval_set=Pool(X_val, y_val))
    proba = model.predict_proba(X_val)[:, 1]
    oof_proba[val_idx] = proba
    pred  = (proba >= THRESHOLD).astype(int)
    cm_f  = confusion_matrix(y_val, pred)
    fold_scores.append({
        'fold': fold,
        'auc':       roc_auc_score(y_val, proba),
        'f1':        f1_score(y_val, pred),
        'recall':    recall_score(y_val, pred),
        'precision': precision_score(y_val, pred),
        'fp':        int(cm_f[0, 1]),
        'best_iter': model.best_iteration_,
    })
    print(f'  Fold {fold} | AUC: {fold_scores[-1]["auc"]:.4f} | '
          f'Recall: {fold_scores[-1]["recall"]:.4f} | '
          f'F1: {fold_scores[-1]["f1"]:.4f} | '
          f'best_iter: {model.best_iteration_}')

scores_df = pd.DataFrame(fold_scores)
print('=' * 65)
print(f"  평균   | AUC: {scores_df.auc.mean():.4f}±{scores_df.auc.std():.4f} "
      f"| Recall: {scores_df.recall.mean():.4f}±{scores_df.recall.std():.4f} "
      f"| F1: {scores_df.f1.mean():.4f}±{scores_df.f1.std():.4f}")

## 6. OOF proba 저장 (.npy)

In [ ]:
os.makedirs(NPY_DIR, exist_ok=True)
npy_path  = os.path.join(NPY_DIR, 'oof_proba_DM_catboost_fe.npy')
oof_array = np.stack([oof_proba, y.values], axis=1)
np.save(npy_path, oof_array)
print(f'저장 완료 → {npy_path}')
loaded = np.load(npy_path)
print(f'로드 확인: shape={loaded.shape}, 일치={np.allclose(oof_array, loaded)}')

## 7. OOF 성능 & 기준 모델 비교

In [ ]:
pred_oof = (oof_proba >= THRESHOLD).astype(int)
cm       = confusion_matrix(y, pred_oof)

oof_auc  = roc_auc_score(y, oof_proba)
oof_rec  = recall_score(y, pred_oof)
oof_prec = precision_score(y, pred_oof)
oof_f1   = f1_score(y, pred_oof)
oof_acc  = float((pred_oof == y).mean())

# 기준 모델 (FE 없음, Optuna)
BASE = {'auc': 0.8091, 'recall': 0.8249, 'precision': 0.2672,
        'f1': 0.4036, 'acc': 0.6723, 'fp': 1835, 'fn': 142}

print('=' * 55)
print(f'  {"지표":<12}  {"기준 모델":>12}  {"FE 적용":>10}  변화')
print('=' * 55)
for label, base_v, cur_v in [
    ('AUC-ROC',   BASE['auc'],       oof_auc ),
    ('Recall',    BASE['recall'],    oof_rec ),
    ('Precision', BASE['precision'], oof_prec),
    ('F1-score',  BASE['f1'],        oof_f1  ),
    ('Accuracy',  BASE['acc'],       oof_acc ),
]:
    d = cur_v - base_v
    arrow = '▲' if d > 0 else ('▼' if d < 0 else '─')
    print(f'  {label:<12}  {base_v:>12.4f}  {cur_v:>10.4f}  {arrow} {abs(d):.4f}')
print(f'  {"FP":<12}  {BASE["fp"]:>12}  {cm[0,1]:>10}  {"▼" if cm[0,1]<BASE["fp"] else "▲"} {abs(cm[0,1]-BASE["fp"])}')
print(f'  {"FN":<12}  {BASE["fn"]:>12}  {cm[1,0]:>10}  {"▼" if cm[1,0]<BASE["fn"] else "▲"} {abs(cm[1,0]-BASE["fn"])}')
print('=' * 55)
print(f'\n[분류 리포트]')
print(classification_report(y, pred_oof, target_names=['정상(0)', '당뇨(1)']))

## 8. Confusion Matrix

In [ ]:
disp = ConfusionMatrixDisplay(cm, display_labels=['정상(0)', '당뇨(1)'])
disp.plot(cmap='Oranges')
plt.title(f'Confusion Matrix — 당뇨 CatBoost FE (threshold={THRESHOLD})')
plt.tight_layout()
plt.show()

## 9. Feature Importance (CatBoost, 마지막 fold)

In [ ]:
fi = pd.DataFrame({'feature': X.columns, 'importance': model.get_feature_importance()})\
       .sort_values('importance', ascending=False)

plt.figure(figsize=(8, 6))
fi.head(20).set_index('feature')['importance'][::-1].plot(kind='barh', color='darkorange')
plt.title('Feature Importance Top 20 — 당뇨 CatBoost FE')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()
print(fi.head(20).to_string(index=False))

## 10. DB 로그 저장

In [ ]:
sys.path.insert(0, os.path.join(BASE_DIR))
from model_logger import ModelLogger

logger = ModelLogger(os.path.join(BASE_DIR, 'model_result.db'))

# 적용된 FE 피처 목록
base_cols   = pd.read_csv(DATA_PATH).columns.tolist()
fe_cols     = [c for c in X.columns if c not in base_cols]
fe_note     = f'FE 적용: {fe_cols}' if fe_cols else 'FE 없음'

run_id = logger.log_run(
    target_var  = '당뇨',
    model_name  = 'CatBoost',
    stage       = f'fe_exp',
    hyperparams = {
        'learning_rate': best_params['learning_rate'],
        'depth': best_params['depth'],
        'n_estimators': best_params['iterations'],
        'class_weight': {0: 1.0, 1: round(ratio, 4)},
        'l2_leaf_reg': best_params['l2_leaf_reg'],
        'bagging_temperature': best_params['bagging_temperature'],
        'random_strength': best_params['random_strength'],
        'border_count': best_params['border_count'],
    },
    data_info = {
        'feature_count': X.shape[1],
        'train_test_split': '5-Fold CV',
        'scaling_method': 'None',
    },
    oof_metrics = {
        'accuracy': oof_acc, 'recall': oof_rec,
        'precision': oof_prec, 'f1_score': oof_f1,
        'auc_roc': oof_auc, 'cm': cm.tolist(),
    },
    fold_scores  = scores_df.to_dict('records'),
    top_features = fi.set_index('feature')['importance'].head(15).to_dict(),
    note = fe_note,
)
print(f'저장 완료 → run_id: {run_id}  |  {fe_note}')
print(logger.compare_runs().to_string(index=False))